In [2]:
# 加载环境
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_community.callbacks.llmonitor_callback import UserContextManager

load_dotenv()

C:\Users\Yangrui\AppData\Local\Temp\ipykernel_10656\858694295.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.callbacks.llmonitor_callback import UserContextManager


True

<div style="font: bold 40px sans-serif;">LangChain Runtime 运行时机制</div>

<br/>
LangChain的Runtime机制是理解Agent内部运行状态的关键。它包括三个核心概念：


| 概念 | 说明                      | 生命周期 |
|-----|-------------------------|---------|
| **State** | 短期记忆，存储Agent当前对话信息、任务状态 | 单次请求 |
| **Store** | 长期记忆，包含用户偏好、失败经验等       | 跨会话 |
| **Context** | 运行时上下文，传递配置参数           | 单次请求 |


# 1.Context(运行时上下文)

Runtime Context 是一种依赖注入机制：调用者在运行开始时把用户身份、租户信息、权限或外部服务连接传给 Agent，节点、中间件和工具在执行时再从 Runtime 读取。

它解决的是三个工程问题：
- 避免把用户 ID、数据库连接等信息硬编码进工具。
- 同一个 Agent 可以安全地服务不同用户和租户。
- 工具的业务参数只保留真正需要模型填写的内容。

## 1.1.准备数据

我们先准备一个字典，模拟一个多租户的在线教育项目数据库，其中的租户就是学校，user就是学员：
- 字典key是租户tenant_id，字典value是租户对应的user信息，也是字典。
- user字典的key是user_id

In [3]:
USER_DATABASE = {
    "school_a": {
        "u001": {"name": "虎哥", "course": "LangChain 开发指南"},
        "u002": {"name": "小明", "course": "Java开发入门"},
    },
    "school_b": {
        "u001": {"name": "涛哥", "course": "Python 入门"},
        "u002": {"name": "阿伟", "course": "VibeCoding 实战"},
    }
}

## 1.2.定义ContextSchema

In [4]:
from dataclasses import dataclass

"""这是一个类型参数，表示当前 Runtime 中的 Context 使用 UserContext 这个数据结构。"""


@dataclass(frozen=True)
class UserContext:
    """Agent运行时上下文"""
    user_id: str
    tenant_id: str


from langchain.tools import tool, ToolRuntime

"""
    LangChain 会在工具执行时自动注入 Runtime 对象。你不需要自己手动创建 runtime，也不需要让大模型生成这个参数。
"""


@tool
def get_current_user_profile(runtime: ToolRuntime[UserContext]):
    """查询当前登录用户的资料"""

    # 获取context，这是整段代码最核心的一行。runtime.context 用于获取当前 Agent 运行时传入的上下文数据。
    ctx = runtime.context

    print("ctx：", ctx)

    # 根据user_id和tenant_id获取用户信息
    profile = USER_DATABASE.get(ctx.tenant_id, {}).get(
        ctx.user_id)  #这里实际上进行了两次字典查询。第一次：根据 tenant_id 获取租户数据，第二次：根据 user_id 获取用户数据

    return f"姓名：{profile['name']}，课程：{profile['course']}"

## 1.4.给Agent添加Context

In [5]:
from langchain.agents import create_agent
import os

qwen_api_key = os.getenv("ALIYUN")
qwen_base_url = os.getenv("dashscope_base_url")

# 创建agent时指定context_schema

qwen_model = init_chat_model(model="qwen-max", model_provider='openai', api_key=qwen_api_key, base_url=qwen_base_url, )

agent = create_agent(model=qwen_model, tools=[get_current_user_profile],
                     system_prompt='你是课程平台助手。需要用户资料时调用工具', context_schema=UserContext, )


## 1.5.调用Agent，传递Context信息

In [6]:
from langchain.messages import HumanMessage

In [20]:

# 调用时通过Context传递用户信息
response = agent.invoke({"messages": [HumanMessage("hello，帮我查询我正在学习的课程")]},
                        context=UserContext(user_id='u001', tenant_id='school_a'), )

for message in response['messages']:
    message.pretty_print()

ctx： UserContext(user_id='u001', tenant_id='school_a')
================================ Human Message =================================

hello，帮我查询我正在学习的课程
================================== Ai Message ==================================
Tool Calls:
  get_current_user_profile (call_b184450792674ec19d6096)
 Call ID: call_b184450792674ec19d6096
  Args:
================================= Tool Message =================================
Name: get_current_user_profile

姓名：虎哥，课程：LangChain 开发指南
================================== Ai Message ==================================

你好，虎哥！你目前正在学习的课程是《LangChain 开发指南》。如果需要更多帮助或资料，请告诉我。


# 2. State（短期记忆）

State是Agent的短期记忆，存储当前会话的历史消息、任务状态等信息。

我们之前学习短期记忆时使用的是默认的AgentState，其中只包含会话的历史消息，本节开始我们学习如何自定义AgentState，记录除了历史消息以外的其他信息。


## 2.1 自定义State

In [7]:
from langchain.agents import AgentState


# 定义 自定义的State结构
class CustomState(AgentState):
    """Agent的任务状态"""
    model_call_count: int  # 用来记录模型调用次数

## 2.2.在工具中访问state

在定义tool的时候，LangChain内置了一个runtime参数，通过runtime我们可以获取Agent的内部信息，包括：
- state
- store
- context

因此，runtime成为了LangChain中tool的限定参数，自定义参数不能叫这个名字。


In [8]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage


@tool
def update_model_call_count(runtime: ToolRuntime):
    """A tool that count model call"""

    # 获取state中的数据
    count = runtime.state.get('model_call_count', 0)
    print(f"model call count: {count}")

    # 更新state中的数据
    update = {
        "model_call_count": count + 1,
        "messages":[
            ToolMessage(
                content="Successfully updated model call count",
                tool_call_id=runtime.tool_call_id
            )
        ]
    }
    return Command(update=update)

In [9]:
# 创建agent
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv
import os
#
# qwen_api_key= os.getenv("ALIYUN")
# qwen_base_url = os.getenv("dashscope_base_url")

# 初始化模型
qwen_model= init_chat_model(
    model="qwen-max",
    model_provider='openai',
    api_key=qwen_api_key,
    base_url=qwen_base_url,
)

# 创建智能体
agent= create_agent(
    model=qwen_model,
    tools=[update_model_call_count],
    system_prompt='你是ai助手，需要在每次回答用户前调用工具记录模型调用次数',
    state_schema=CustomState,
    checkpointer=InMemorySaver(),
)

In [36]:
from langchain.messages import HumanMessage

# 添加配置
config={
    "configurable":{
        "thread_id":"user_001",
    }
}

# 调用模型
response = agent.invoke({"messages": [HumanMessage("你好，我是云深不知处")]}, config=config)

# for chunk in response['messages']:
    # print(chunk.)

model call count: 1


In [37]:
for chunk in response['messages']:
    print(chunk.pretty_print())

================================ Human Message =================================

你好，我是云深不知处
None
================================== Ai Message ==================================
Tool Calls:
  update_model_call_count (call_c73616c03b954626908404)
 Call ID: call_c73616c03b954626908404
  Args:
None
================================= Tool Message =================================
Name: update_model_call_count

Successfully updated model call count
None
================================== Ai Message ==================================

你好，云深不知处！有什么可以帮助你的吗？
None
================================ Human Message =================================

你好，我是云深不知处
None
================================== Ai Message ==================================
Tool Calls:
  update_model_call_count (call_b3069ae03b024fbbaca399)
 Call ID: call_b3069ae03b024fbbaca399
  Args:
None
================================= Tool Message =================================
Name: update_model_call_count

Successfully updated model ca

In [38]:
# 来获取最后的state的快照
print(agent.get_state(config).values['model_call_count'])

2


# 3. Store（长期记忆）

store是LangChain提供的长期记忆机制，用于在不同会话间共享数据。例如：模型以外的数据、用户偏好等。
LangChain提供了多种Store的实现方式，例如：
- InMemoryStore
- PostgresStore
- RedisStore
- ...

课程中我们以InMemoryStore为例来学习。


## 3.1 初始化Store

In [1]:
from langgraph.store.memory import InMemoryStore

memory_store = InMemoryStore()


## 3.2 在tool中访问store

In [21]:
from langchain.tools import ToolRuntime,tool

# 定义方法，根据context中的tenant_id和user_id拼接得到namespace
def preference_namespace(ctx:UserContext) -> tuple[str,...]:
    return (
        "heima_platform",
        ctx.tenant_id,
        ctx.user_id
    )

# 定义固定的key
key="learning_preference"       # 表示学习偏好

@tool
def save_learning_preference(
        preference_name:str,
        preference_value:str,
        runtime:ToolRuntime[UserContext]
) -> str:
    """保存当前用户的长期学习偏好"""

    # 通过上下文中的tenant_id和user_id来构建namespace
    namespace = preference_namespace(runtime.context)
    print("namespace：",namespace)

    # 通过namespace和key来读取用户偏好，返回结果item.value 是dict格式
    item = runtime.store.get(namespace, key)
    preferences = item.value if item else {}

    # 记录本次要保存的用户偏好
    preferences[preference_name] = preference_value

    # 保存到store中
    runtime.store.put(namespace, key, preferences)

    return f"保存成功，已保存偏好：{preference_name}={preference_value}"


In [22]:
@tool
def get_learning_preference(
        runtime:ToolRuntime[UserContext]
) -> str:
    """获取当前用户的长期学习偏好"""
    namespace = preference_namespace(runtime.context)
    item = runtime.store.get(namespace, key)

    if item is None:
        return "当前用户还没有保存学习偏好"

    return f"当前用户的学习偏好为：{item.value}"

## 3.3.创建Agent，指定Store

In [23]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langchain.chat_models import init_chat_model

checkpointer = InMemorySaver()
store = InMemoryStore()

# 前面已经创建过qwenModel了，这里直接使用
agent = create_agent(
    model=qwen_model,
    tools=[save_learning_preference,get_learning_preference,get_current_user_profile],
    system_prompt='你是课程平台助手。涉及用户资料时调用资料工具。涉及长期学习偏好或给用户讲解代码时调用偏好工具',
    checkpointer=checkpointer,
    store=store,
)

In [24]:
context = UserContext(
    user_id="u001",
    tenant_id="school_a",
)

result = agent.invoke(
    {"messages": [HumanMessage("记住：我喜欢先看代码，再看原理")]},
    config={"configurable": {"thread_id": "thread-a"}},
    context=context,
)
for message in result["messages"]:
    message.pretty_print()

namespace： ('heima_platform', 'school_a', 'u001')
================================ Human Message =================================

记住：我喜欢先看代码，再看原理
================================== Ai Message ==================================
Tool Calls:
  save_learning_preference (call_122c7b8b130b40b0a82c67)
 Call ID: call_122c7b8b130b40b0a82c67
  Args:
    preference_name: learning_style
    preference_value: code_first_then_principle
================================= Tool Message =================================
Name: save_learning_preference

保存成功，已保存偏好：learning_style=code_first_then_principle
================================== Ai Message ==================================

好的，我已经记住了您的偏好。以后我会先向您展示代码，然后再解释背后的原理。如果您还有其他学习偏好想要设置，请随时告诉我！


In [26]:
# 这次传了一个新的thread_id，之前传的是thread-a，这次是thread-b，所以是在不同的会话窗口中

result = agent.invoke(
    {"messages": [HumanMessage("langchain如何实现结构化输出")]},
    config={"configurable": {"thread_id": "thread-b"}},
    context=context,
)
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

langchain如何实现结构化输出
================================== Ai Message ==================================

LangChain 是一个用于开发由语言模型驱动的应用程序的框架，它可以帮助开发者更方便地利用大型语言模型的能力。要使用 LangChain 实现结构化输出，通常涉及以下步骤：

1. **定义输出模式（Output Schema）**:
   你需要首先定义你期望的输出格式。这可以通过 Python 的数据类（dataclasses）、Pydantic 模型或其他方式来完成。

2. **创建链（Chain）或代理（Agent）**:
   使用 LangChain 创建一个处理输入并生成符合你所定义模式的输出的链或代理。

3. **使用输出解析器（Output Parser）**:
   如果你的链或代理产生的文本需要被转换成结构化数据，那么可以使用输出解析器将自然语言输出转换为你定义的数据结构。

4. **执行和验证**:
   执行链或代理，并确保输出与预期的结构相匹配。如果不符合，可能需要调整提示（Prompt）或者输出解析逻辑。

下面是一个简单的例子，演示如何在 LangChain 中实现结构化输出。假设我们要从一段文本中提取人物的名字、年龄和职业信息。

### 示例代码

```python
from langchain import LLMChain, PromptTemplate
from langchain.llms import OpenAI
from pydantic import BaseModel, Field
from typing import List
import json

# 定义输出模式
class PersonInfo(BaseModel):
    name: str = Field(..., description="The person's name")
    age: int = Field(..., description="The perso